# Cell2Cell Telecom — Visualizations


1. **EDA Overview** — churn distributions, key feature patterns
2. **CLV Clustering** — elbow chart, silhouette, tier distributions
3. **Model Evaluation** — ROC, precision-recall, confusion matrix, CV scores
4. **SHAP Explainability** — beeswarm and bar chart
5. **Retention ROI** — segment map, revenue at risk, ROI comparison
6. **Customer Deep Dive** — individual Priority Save customer analysis
7. **Holdout Validation** — unseen 20K customer predictions


## Setup

In [2]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import shap

from sklearn.metrics import roc_curve, confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve

FIG = './figures'
os.makedirs(FIG, exist_ok=True)

TEAL   = '#1D9E75'
CORAL  = '#D85A30'
PURPLE = '#7F77DD'
AMBER  = '#BA7517'
BLUE   = '#378ADD'
GRAY   = '#888780'
GREEN  = '#639922'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.color': '#cccccc',
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

seg_colors = {
    'Priority Save': CORAL,
    'Consider Save': AMBER,
    'Let Go': GRAY,
    'Nurture': TEAL,
    'Monitor': BLUE
}

print('done')

done


In [7]:
# load all results 
with open('payload.pkl', 'rb') as f:
    p = pickle.load(f)

df          = p['df'];        df_raw     = p['df_raw']
X           = p['X'];         y          = p['y']
X_te        = p['X_te'];      y_te       = p['y_te']
y_pred      = p['y_pred'];    y_prob     = p['y_prob']
auc         = p['auc'];       f1_s       = p['f1'];     ap = p['ap']
cv          = p['cv']
shap_s      = p['shap_s'];    sv         = p['sv']
smart_net   = p['smart_net']; naive_net  = p['naive_net']
sg          = p['sg'];        ng         = p['ng']
OFFER       = p['OFFER'];     SR         = p['SR'];     MO = p['MO']
sil         = p['sil']
inertias    = p['inertias'];  sil_scores = p['sil_scores']
y_prob_hold = p['y_prob_hold']; df_hold  = p['df_hold']
model       = p['model']

print('done')

done


## Figure 1 — EDA Overview

In [9]:
fig = plt.figure(figsize=(18, 13))
fig.suptitle('Cell2Cell Telecom — EDA Overview  (51,047 customers · 58 features)',
             fontsize=15, fontweight='bold', y=0.99)
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.52, wspace=0.38)

# Churn distribution
ax = fig.add_subplot(gs[0, 0])
vals = [(df_raw['Churn'] == 'No').sum(), (df_raw['Churn'] == 'Yes').sum()]
bars = ax.bar(['No Churn', 'Churn'], vals, color=[TEAL, CORAL], width=0.5, edgecolor='white')
ax.set_title('Overall churn distribution', fontweight='bold')
ax.set_ylabel('Customers')
for b in bars:
    pct = b.get_height() / len(df_raw)
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 300,
            f'{b.get_height():,}\n({pct:.1%})', ha='center', fontsize=10, fontweight='bold')
ax.set_ylim(0, max(vals) * 1.25)

# Monthly Revenue by Churn
ax = fig.add_subplot(gs[0, 1])
rev_n = df_raw[df_raw['Churn'] == 'No']['MonthlyRevenue'].dropna()
rev_y = df_raw[df_raw['Churn'] == 'Yes']['MonthlyRevenue'].dropna()
ax.hist(rev_n, bins=50, alpha=0.65, color=TEAL,  label=f'No Churn (med=${rev_n.median():.0f})', density=True)
ax.hist(rev_y, bins=50, alpha=0.65, color=CORAL, label=f'Churn (med=${rev_y.median():.0f})', density=True)
ax.set_title('Monthly revenue by churn', fontweight='bold')
ax.set_xlabel('Monthly Revenue ($)'); ax.set_ylabel('Density')
ax.legend(fontsize=8)

#  Months in Service
ax = fig.add_subplot(gs[0, 2])
mon_n = df_raw[df_raw['Churn'] == 'No']['MonthsInService'].dropna()
mon_y = df_raw[df_raw['Churn'] == 'Yes']['MonthsInService'].dropna()
ax.hist(mon_n, bins=40, alpha=0.65, color=TEAL,  label=f'No Churn (med={mon_n.median():.0f})', density=True)
ax.hist(mon_y, bins=40, alpha=0.65, color=CORAL, label=f'Churn (med={mon_y.median():.0f})', density=True)
ax.set_title('Months in service by churn', fontweight='bold')
ax.set_xlabel('Months'); ax.legend(fontsize=8)

# Customer Care Calls vs churn rate
ax = fig.add_subplot(gs[0, 3])
tmp = df_raw.copy()
tmp['Churn_num'] = (tmp['Churn'] == 'Yes').astype(float)
cc = tmp.groupby('CustomerCareCalls')['Churn_num'].mean() * 100
cc = cc[cc.index <= 10].reset_index()
ax.bar(cc['CustomerCareCalls'], cc['Churn_num'],
       color=[CORAL if v > 35 else AMBER if v > 25 else TEAL for v in cc['Churn_num']],
       edgecolor='white')
ax.set_title('Churn rate vs care calls', fontweight='bold')
ax.set_xlabel('Customer care calls'); ax.set_ylabel('Churn rate (%)')

# Dropped Calls violin
ax = fig.add_subplot(gs[1, 0])
dc_n = df_raw[df_raw['Churn'] == 'No']['DroppedCalls'].clip(0, 20).dropna()
dc_y = df_raw[df_raw['Churn'] == 'Yes']['DroppedCalls'].clip(0, 20).dropna()
vp = ax.violinplot([dc_n.values, dc_y.values], positions=[1, 2], showmedians=True, showextrema=False)
for pc in vp['bodies']:
    pc.set_alpha(0.65)
ax.set_xticks([1, 2]); ax.set_xticklabels(['No Churn', 'Churn'])
ax.set_title('Dropped calls distribution', fontweight='bold'); ax.set_ylabel('Dropped calls')

# Retention Calls vs churn rate
ax = fig.add_subplot(gs[1, 1])
rc = tmp.groupby('RetentionCalls')['Churn_num'].mean() * 100
rc = rc[rc.index <= 8].reset_index()
ax.plot(rc['RetentionCalls'], rc['Churn_num'], 'o-', color=PURPLE, lw=2, markersize=7)
ax.fill_between(rc['RetentionCalls'], rc['Churn_num'], alpha=0.12, color=PURPLE)
ax.set_title('Churn rate vs retention calls', fontweight='bold')
ax.set_xlabel('Retention calls'); ax.set_ylabel('Churn rate (%)')

#  Credit Rating vs churn
ax = fig.add_subplot(gs[1, 2])
cr = tmp.groupby('CreditRating')['Churn_num'].mean() * 100
cr = cr.dropna().sort_values(ascending=True).reset_index()
ax.barh(cr['CreditRating'].astype(str), cr['Churn_num'],
        color=[CORAL if v > 32 else AMBER if v > 25 else TEAL for v in cr['Churn_num']],
        edgecolor='white')
ax.set_title('Churn rate by credit rating', fontweight='bold'); ax.set_xlabel('Churn rate (%)')

# Overage Minutes
ax = fig.add_subplot(gs[1, 3])
ov_n = df_raw[df_raw['Churn'] == 'No']['OverageMinutes'].clip(0, 200).dropna()
ov_y = df_raw[df_raw['Churn'] == 'Yes']['OverageMinutes'].clip(0, 200).dropna()
ax.hist(ov_n, bins=40, alpha=0.65, color=TEAL,  label='No Churn', density=True)
ax.hist(ov_y, bins=40, alpha=0.65, color=CORAL, label='Churn',    density=True)
ax.set_title('Overage minutes by churn', fontweight='bold')
ax.set_xlabel('Overage minutes'); ax.legend(fontsize=8)

#  Occupation vs churn
ax = fig.add_subplot(gs[2, 0])
occ = tmp.groupby('Occupation')['Churn_num'].mean() * 100
occ = occ.dropna().sort_values(ascending=True).reset_index()
ax.barh(occ['Occupation'].astype(str), occ['Churn_num'],
        color=[CORAL if v > 30 else AMBER if v > 24 else TEAL for v in occ['Churn_num']],
        edgecolor='white', height=0.6)
ax.set_title('Churn rate by occupation', fontweight='bold'); ax.set_xlabel('Churn rate (%)')

ax = fig.add_subplot(gs[2, 1])
pz = tmp.groupby('PrizmCode')['Churn_num'].mean() * 100
pz = pz.dropna().sort_values(ascending=False).reset_index()
ax.bar(pz['PrizmCode'].astype(str), pz['Churn_num'],
       color=[CORAL, AMBER, TEAL, BLUE][:len(pz)], edgecolor='white', width=0.5)
ax.set_title('Churn rate by area type', fontweight='bold'); ax.set_ylabel('Churn rate (%)')

# Marital Status vs churn
ax = fig.add_subplot(gs[2, 2])
ms = tmp.groupby('MaritalStatus')['Churn_num'].mean() * 100
ms = ms.dropna().reset_index()
ax.bar(ms['MaritalStatus'].astype(str), ms['Churn_num'],
       color=[TEAL, CORAL, AMBER][:len(ms)], edgecolor='white', width=0.5)
ax.set_title('Churn by marital status', fontweight='bold'); ax.set_ylabel('Churn rate (%)')

# Correlation heatmap
ax = fig.add_subplot(gs[2, 3])
corr_cols = ['MonthlyRevenue', 'MonthlyMinutes', 'DroppedCalls',
             'CustomerCareCalls', 'MonthsInService', 'RetentionCalls']
df_cn = tmp[corr_cols + ['Churn_num']].copy().rename(columns={
    'MonthlyRevenue': 'Revenue', 'MonthlyMinutes': 'Minutes',
    'DroppedCalls': 'Dropped', 'CustomerCareCalls': 'CustCare',
    'MonthsInService': 'Tenure', 'RetentionCalls': 'RetCalls', 'Churn_num': 'Churn'
})
sns.heatmap(df_cn.corr(), annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            ax=ax, cbar=False, linewidths=0.5, annot_kws={'size': 7})
ax.set_title('Correlation matrix', fontweight='bold'); ax.tick_params(labelsize=7)

plt.savefig(f'{FIG}/fig1_eda_overview.png', dpi=150, bbox_inches='tight')
plt.close()
print('done')

done


## Figure 2 — CLV Clustering

In [10]:
tier_col = {'Low CLV': CORAL, 'Mid CLV': AMBER, 'High CLV': TEAL}
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('CLV Clustering Analysis — Cell2Cell (k=3 K-Means)', fontsize=14, fontweight='bold')

K_range = range(2, 9)
axes[0, 0].plot(list(K_range), inertias, 'o-', color=CORAL, lw=2, markersize=7)
axes[0, 0].axvline(3, color=TEAL, linestyle='--', lw=1.5, label='Chosen k=3')
axes[0, 0].set_title('Elbow method', fontweight='bold')
axes[0, 0].set_xlabel('K'); axes[0, 0].set_ylabel('Inertia'); axes[0, 0].legend()

axes[0, 1].bar(list(K_range), sil_scores,
               color=[TEAL if k == 3 else GRAY for k in K_range], edgecolor='white')
axes[0, 1].set_title('Silhouette scores by K', fontweight='bold')
axes[0, 1].set_xlabel('K'); axes[0, 1].set_ylabel('Silhouette')

for tier, col in tier_col.items():
    d = df[df['CLV_tier'] == tier]['CLV']
    axes[1, 0].hist(d, bins=60, alpha=0.6, color=col,
                    label=f'{tier} (n={len(d):,}, avg=${d.mean():,.0f})', density=True)
axes[1, 0].set_title('CLV distribution by tier', fontweight='bold')
axes[1, 0].set_xlabel('CLV ($)'); axes[1, 0].set_ylabel('Density'); axes[1, 0].legend(fontsize=8)

samp = df.sample(6000, random_state=42)
for tier, col in tier_col.items():
    m = samp['CLV_tier'] == tier
    axes[1, 1].scatter(samp[m]['MonthsInService'], samp[m]['MonthlyRevenue'],
                       c=col, alpha=0.25, s=8, label=tier)
axes[1, 1].set_title('Tenure vs monthly revenue (sample)', fontweight='bold')
axes[1, 1].set_xlabel('Months in service'); axes[1, 1].set_ylabel('Monthly Revenue ($)')
axes[1, 1].legend(markerscale=3, fontsize=8)

plt.tight_layout()
plt.savefig(f'{FIG}/fig2_clv_clustering.png', dpi=150, bbox_inches='tight')
plt.close()
print('done')

done


## Figure 3 — Model Evaluation

In [11]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Model Evaluation — XGBoost Churn Classifier (Cell2Cell)', fontsize=14, fontweight='bold')

# ROC curve
fpr, tpr, _ = roc_curve(y_te, y_prob)
axes[0, 0].plot(fpr, tpr, color=TEAL, lw=2.5, label=f'XGBoost (AUC={auc:.3f})')
axes[0, 0].plot([0, 1], [0, 1], 'k--', alpha=0.3, lw=1)
axes[0, 0].fill_between(fpr, tpr, alpha=0.08, color=TEAL)
axes[0, 0].set_title('ROC Curve', fontweight='bold')
axes[0, 0].set_xlabel('False Positive Rate'); axes[0, 0].set_ylabel('True Positive Rate')
axes[0, 0].legend(loc='lower right')

# Precision-Recall
prec, rec, _ = precision_recall_curve(y_te, y_prob)
axes[0, 1].plot(rec, prec, color=PURPLE, lw=2.5, label=f'AP={ap:.3f}')
axes[0, 1].axhline(y_te.mean(), color=GRAY, linestyle='--', lw=1.5,
                   label=f'Baseline ({y_te.mean():.2f})')
axes[0, 1].fill_between(rec, prec, alpha=0.08, color=PURPLE)
axes[0, 1].set_title('Precision-Recall Curve', fontweight='bold')
axes[0, 1].set_xlabel('Recall'); axes[0, 1].set_ylabel('Precision'); axes[0, 1].legend()

# Confusion Matrix
cm = confusion_matrix(y_te, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['No Churn', 'Churn']).plot(
    ax=axes[0, 2], colorbar=False, cmap='Blues')
axes[0, 2].set_title('Confusion Matrix', fontweight='bold')
tn, fp, fn, tp = cm.ravel()
axes[0, 2].text(0.5, -0.18, f'TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}',
                transform=axes[0, 2].transAxes, ha='center', fontsize=9, color=GRAY)

# CV scores
axes[1, 0].bar([f'Fold {i+1}' for i in range(5)], cv,
               color=[TEAL if s >= cv.mean() else CORAL for s in cv], edgecolor='white')
axes[1, 0].axhline(cv.mean(), color='black', linestyle='--', lw=1.5,
                   label=f'Mean={cv.mean():.3f}')
axes[1, 0].set_title('5-Fold CV AUC', fontweight='bold')
axes[1, 0].set_ylabel('ROC-AUC'); axes[1, 0].legend(); axes[1, 0].set_ylim(0.5, 0.9)

# Probability distribution
axes[1, 1].hist(y_prob[y_te == 0], bins=50, alpha=0.7, color=TEAL,  label='Actual: No Churn', density=True)
axes[1, 1].hist(y_prob[y_te == 1], bins=50, alpha=0.7, color=CORAL, label='Actual: Churn', density=True)
axes[1, 1].axvline(0.5, color='black', linestyle='--', lw=1.5, label='Threshold=0.5')
axes[1, 1].set_title('Predicted churn probability distribution', fontweight='bold')
axes[1, 1].set_xlabel('Predicted churn probability'); axes[1, 1].legend(fontsize=8)

# Feature importance
fi = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True).tail(15)
axes[1, 2].barh(fi.index, fi.values,
                color=[CORAL if f in fi.tail(5).index else BLUE for f in fi.index],
                edgecolor='white')
axes[1, 2].set_title('Top 15 features (XGBoost gain)', fontweight='bold')
axes[1, 2].set_xlabel('Importance')

plt.tight_layout()
plt.savefig(f'{FIG}/fig3_model_evaluation.png', dpi=150, bbox_inches='tight')
plt.close()
print('done')

done


## Figure 4 — SHAP Explainability

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('SHAP Explainability — Why Does the Model Predict Churn?', fontsize=14, fontweight='bold')

plt.sca(axes[0])
shap.summary_plot(sv, shap_s, max_display=15, show=False, plot_size=None)
axes[0].set_title('SHAP beeswarm — top 15 features\n(red = pushes toward churn, blue = away from churn)',
                  fontweight='bold', pad=10)

feat_shap = pd.Series(np.abs(sv).mean(axis=0),
                      index=shap_s.columns).sort_values(ascending=True).tail(15)
axes[1].barh(feat_shap.index, feat_shap.values,
             color=[CORAL if f in feat_shap.tail(5).index else BLUE for f in feat_shap.index],
             edgecolor='white')
axes[1].set_title('Mean |SHAP value| — global feature importance', fontweight='bold')
axes[1].set_xlabel('Mean |SHAP value|')

plt.tight_layout()
plt.savefig(f'{FIG}/fig4_shap_explainability.png', dpi=150, bbox_inches='tight')
plt.close()
print('done')

done


## Figure 5 — Retention Matrix & ROI

In [13]:
thresholds = np.arange(0.3, 0.76, 0.05)
smart_t, naive_t = [], []
for t in thresholds:
    sg_t = df[(df['churn_prob'] >= t) & (df['CLV_tier'].isin(['High CLV', 'Mid CLV']))]
    ng_t = df[df['churn_prob'] >= t]
    smart_t.append(sg_t['MonthlyRevenue'].sum() * SR * MO - len(sg_t) * OFFER)
    naive_t.append(ng_t['MonthlyRevenue'].sum() * SR * MO - len(ng_t) * OFFER)

fig = plt.figure(figsize=(18, 13))
fig.suptitle('Retention Matrix & ROI Simulation — Cell2Cell (51,047 customers)',
             fontsize=14, fontweight='bold')
gs2 = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.36)

# Segment pie
ax = fig.add_subplot(gs2[0, 0])
sc_ = df['segment'].value_counts()
wedges, texts, autotexts = ax.pie(
    sc_.values, labels=sc_.index, autopct='%1.1f%%',
    colors=[seg_colors[s] for s in sc_.index], startangle=90, pctdistance=0.72
)
for t in autotexts:
    t.set_fontsize(8)
ax.set_title(f'Customer segments (n={len(df):,})', fontweight='bold')

# Revenue at risk
ax = fig.add_subplot(gs2[0, 1])
rev_risk = (df[df['churn_pred'] == 1].groupby('segment')['MonthlyRevenue']
            .sum().sort_values(ascending=True))
ax.barh(rev_risk.index, rev_risk.values,
        color=[seg_colors.get(s, GRAY) for s in rev_risk.index], edgecolor='white')
ax.set_title('Monthly revenue at risk\n(predicted churners)', fontweight='bold')
ax.set_xlabel('Monthly revenue ($)')
for i, (idx, v) in enumerate(rev_risk.items()):
    ax.text(v + 200, i, f'${v:,.0f}', va='center', fontsize=8)

# ROI comparison
ax = fig.add_subplot(gs2[0, 2])
bars = ax.bar(['Naive\n(all churners)', 'Smart\n(CLV-filtered)'],
              [naive_net, smart_net], color=[GRAY, TEAL], edgecolor='white', width=0.45)
ax.set_title('Annual net ROI comparison', fontweight='bold'); ax.set_ylabel('Net ROI ($/year)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() * 1.02,
            f'${b.get_height()/1e6:.2f}M', ha='center', fontsize=11, fontweight='bold')

# Retention scatter
ax = fig.add_subplot(gs2[1, :2])
samp2 = df.sample(8000, random_state=42)
for seg, col in seg_colors.items():
    m = samp2['segment'] == seg
    ax.scatter(samp2[m]['churn_prob'], samp2[m]['CLV'],
               c=col, alpha=0.3, s=8, label=f'{seg} ({m.sum():,})')
ax.axvline(0.5, color='black', linestyle='--', lw=1.2, alpha=0.4)
ax.axhline(df['CLV'].median(), color='black', linestyle='--', lw=1.2, alpha=0.4)
ax.set_xlabel('Churn probability'); ax.set_ylabel('CLV ($)')
ax.set_title('Retention matrix — 8,000 customer sample', fontweight='bold')
ax.legend(markerscale=3, fontsize=8, loc='upper left')

# Threshold sensitivity
ax = fig.add_subplot(gs2[1, 2])
ax.plot(thresholds, [v / 1e6 for v in smart_t], 'o-', color=TEAL, lw=2, label='Smart strategy')
ax.plot(thresholds, [v / 1e6 for v in naive_t],  's--', color=GRAY, lw=2, label='Naive strategy')
ax.axvline(0.5, color=AMBER, linestyle=':', lw=1.5, label='Default threshold (0.5)')
ax.set_title('ROI vs decision threshold', fontweight='bold')
ax.set_xlabel('Churn probability threshold'); ax.set_ylabel('Annual net ROI ($M)')
ax.legend(fontsize=9)

plt.savefig(f'{FIG}/fig5_retention_roi.png', dpi=150, bbox_inches='tight')
plt.close()
print('done')

done


## Figure 6 — Customer Deep Dive

In [14]:
test_meta = df.loc[X_te.index]
ps_indices = np.where((test_meta['segment'] == 'Priority Save').values)[0]
sample_pos = ps_indices[0] if len(ps_indices) > 0 else 0

s_prob = y_prob[sample_pos]
s_shap = sv[shap_s.index.get_loc(X_te.index[sample_pos])] if X_te.index[sample_pos] in shap_s.index else sv[0]
s_clv  = df.loc[X_te.index[sample_pos], 'CLV']
s_rev  = X_te.iloc[sample_pos].get('MonthlyRevenue', 65)

top_idx   = np.argsort(np.abs(s_shap))[-10:][::-1]
top_names = X_te.columns[top_idx]
top_vals  = s_shap[top_idx]
top_data  = X_te.iloc[sample_pos][top_names].values

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('Priority Save Customer Deep Dive — Cell2Cell', fontsize=14, fontweight='bold')

# SHAP waterfall for one customer
colors_wf = [CORAL if v > 0 else TEAL for v in top_vals]
axes[0].barh(range(10), top_vals[::-1], color=colors_wf[::-1], edgecolor='white', height=0.6)
axes[0].set_yticks(range(10))
axes[0].set_yticklabels(
    [f'{top_names[9-i]}  (val={top_data[9-i]:.1f}, shap={top_vals[9-i]:+.3f})' for i in range(10)],
    fontsize=8)
axes[0].axvline(0, color='black', lw=0.8, alpha=0.4)
axes[0].set_xlabel('SHAP value (impact on churn probability)')
axes[0].set_title(f'Why {s_prob:.0%} churn probability?\nTop 10 feature contributions', fontweight='bold')

# Business action brief
axes[1].axis('off')
brief = f"""
CUSTOMER PROFILE
{'─'*42}
 Churn probability :  {s_prob:.1%}
 CLV score         :  ${s_clv:,.2f}
 Monthly revenue   :  ${s_rev:,.2f}
 Segment           :  PRIORITY SAVE

{'─'*42}
TOP CHURN DRIVERS

"""
for name, val, sv_ in zip(top_names[:6], top_data[:6], top_vals[:6]):
    arrow = '▲ RISK' if sv_ > 0 else '▼ SAFE'
    brief += f'  {arrow}  {name:28s}  {sv_:+.3f}\n'

brief += f"""
{'─'*42}
RECOMMENDED ACTION

  Assign to retention team immediately.
  Estimated annual value :  ${s_clv:,.0f}
  Suggested offer cap    :  ${s_rev*3:,.0f}
  (3-month payback period)

  Without action — expected annual loss:
  ${s_rev*12:,.0f} / year
{'─'*42}
"""
axes[1].text(0.04, 0.97, brief, transform=axes[1].transAxes,
             fontsize=10, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='#fff9f0',
                       alpha=0.95, edgecolor=CORAL, linewidth=2))
axes[1].set_title('Business action brief', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{FIG}/fig6_customer_deep_dive.png', dpi=150, bbox_inches='tight')
plt.close()
print('done')

done


## Figure 7 — Holdout Validation

In [16]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Holdout Set Validation — 20,000 Unseen Customers', fontsize=14, fontweight='bold')

# Probability distribution
axes[0].hist(y_prob_hold, bins=60, color=PURPLE, alpha=0.8, edgecolor='white')
axes[0].axvline(0.5, color=CORAL, linestyle='--', lw=2, label='Threshold=0.5')
pct_h = (y_prob_hold >= 0.5).mean()
axes[0].set_title('Predicted churn probabilities\n(holdout — 20K customers)', fontweight='bold')
axes[0].set_xlabel('Churn probability'); axes[0].set_ylabel('Count'); axes[0].legend()
axes[0].text(0.52, axes[0].get_ylim()[1] * 0.8, f'{pct_h:.1%}\nflagged', fontsize=11, color=CORAL)

# Revenue distribution
hold_rev = df_hold['MonthlyRevenue'] if 'MonthlyRevenue' in df_hold.columns else pd.Series(np.zeros(len(df_hold)))
at_risk  = hold_rev[y_prob_hold >= 0.5]
axes[1].hist(hold_rev, bins=60, color=GRAY,  alpha=0.6, label='All customers', density=True)
axes[1].hist(at_risk,  bins=60, color=CORAL, alpha=0.7, label='Predicted churners', density=True)
axes[1].set_title('Revenue: all vs predicted churners\n(holdout)', fontweight='bold')
axes[1].set_xlabel('Monthly Revenue ($)'); axes[1].legend(fontsize=9)

# Probability buckets
buckets = pd.cut(y_prob_hold, bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
                 labels=['0–20%', '20–40%', '40–60%', '60–80%', '80–100%'])
bc = buckets.value_counts().sort_index()
axes[2].bar(bc.index, bc.values, color=[TEAL, TEAL, AMBER, CORAL, CORAL], edgecolor='white')
axes[2].set_title('Customers by churn\nprobability bucket (holdout)', fontweight='bold')
axes[2].set_xlabel('Churn probability range'); axes[2].set_ylabel('Customers')
for i, v in enumerate(bc.values):
    axes[2].text(i, v + 80, f'{v:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIG}/fig7_holdout_validation.png', dpi=150, bbox_inches='tight')
plt.close()
print('done')

print('\nAll 7 figures saved to the figures')

done

All 7 figures saved to the figures
